# 03 · Templates & Variables

> **Module 1 · Composer Fundamentals & Model · Lecture 1.5–1.6**

In this notebook you will:

- Render Jinja templates the way Airflow does, with **no Airflow installed**
- Enumerate **every** context variable and what it is actually for
- See the undefined-variable failure — and why it is a feature, not a bug
- Separate **Variables**, **params** and **Connections**, and know which to use
- Build the layering pattern: hard-coded default → Variable → param

## 1. Why templating exists

A DAG file is Python code that runs at **parse time**. But many values in a
DAG are only known at **run time**: the logical date, the bucket name, the
partition, the value another task computed.

Templating bridges that gap. You write a *string containing Jinja syntax* in
a place Airflow knows to render, and Airflow substitutes run-time values
before the task executes.

```python
bash_command="gsutil cp gs://{{ var.value.source_bucket }}/{{ ds }}/orders.csv gs://{{ var.value.target_bucket }}/"
```

At parse time that is just a string. At run time it becomes:

```bash
gsutil cp gs://demo-source-bucket/2026-01-15/orders.csv gs://demo-target-bucket/
```

This is what lets **one DAG file serve every environment and every date**.

### The four rules

1. The context is a plain dict.
2. Templates are strings.
3. Rendering happens **before** the task body runs.
4. An undefined variable is a **hard error**, not an empty string.

The course reimplements Airflow's rendering with plain Jinja2 in
`project/core/templating.py` — a *teaching model*, not a replacement. Airflow's
real context is richer and adds `macros`, `task_instance`, and per-operator
isolation. But the mechanics are identical, and you can now debug templates
without a scheduler.

## 2. The context — every variable Airflow gives you

Airflow builds a fresh context **per task, per DAG run**, and hands it to
Jinja2. `build_context()` reproduces it exactly.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

from templating import build_context

context = build_context(
    dag_id="etl_to_gcs",
    task_id="aggregate",
    logical_date="2026-01-15",
)

print(f"context for dag={context['dag']['dag_id']} task={context['task']['task_id']}")
print(f"logical_date={context['logical_date']}\n")

print("--- Date / time templates ---")
for key in (
    "ds", "ts", "ds_nodash", "prev_ds", "next_ds",
    "data_interval_start", "data_interval_end", "logical_date", "run_id",
):
    print(f"  {key:<22} {context[key]}")

print()
print("--- Object graph ---")
for key in ("dag", "task", "ti"):
    print(f"  {key:<22} {context[key]}")

### The context reference

| Name | Example value | What it is |
|---|---|---|
| `ds` | `2026-01-15` | logical date as `YYYY-MM-DD` |
| `ts` | `2026-01-15T00:00:00+00:00` | logical date as a full timestamp |
| `ds_nodash` | `20260115` | logical date without dashes |
| `prev_ds` | `2026-01-14` | previous interval's start |
| `next_ds` | `2026-01-16` | next interval's start |
| `data_interval_start` | `2026-01-15T00:00:00+00:00` | interval start (inclusive) |
| `data_interval_end` | `2026-01-16T00:00:00+00:00` | interval end (**exclusive**) |
| `logical_date` | `2026-01-15T00:00:00+00:00` | the run's label |
| `run_id` | `manual__2026-01-15T00:00:00+00:00` | the run's identifier |
| `dag` | `<DAG: etl_to_gcs>` | the DAG object |
| `task` | `<Task: aggregate>` | the task object |
| `ti` | `<TaskInstance: ...>` | the task instance |
| `var` | — | Airflow Variables |
| `conn` | — | Airflow Connections |
| `params` | — | DAG-run parameters |

### `prev_ds` is one *tick* back, not one day back

This trips people up constantly. For a `@daily` DAG, `prev_ds` is one day
back. For a `timedelta(hours=6)` DAG, it is **six hours** back. `prev_ds` is
"the previous schedule tick", and it is the right thing to use when you want
the previous *interval* — because the previous interval's length matches your
schedule's cadence.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

from templating import build_context

print("How prev_ds relates to the schedule cadence:")
print()
for days in (1, 7):
    ctx = build_context(logical_date="2026-01-15", data_interval_days=days)
    print(f"  data_interval_days={days}")
    print(f"    ds                  = {ctx['ds']}")
    print(f"    prev_ds             = {ctx['prev_ds']}")
    print(f"    data_interval_start = {ctx['data_interval_start']}")
    print(f"    data_interval_end   = {ctx['data_interval_end']}")
    print()

# The half-open window is directly visible in the context.
ctx = build_context(logical_date="2026-01-15")
print("Half-open interval, straight from the context:")
print(f"  [{ctx['data_interval_start']}, {ctx['data_interval_end']})")
print()
print("  start is INCLUDED, end is EXCLUDED.")

## 3. Rendering templates

`render_template(template, context)` is the one call you need. It uses
`StrictUndefined`, so an unknown name is an error — as it is in Airflow.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

from templating import render_template

TEMPLATES = {
    "source": "gs://{{ var.value.source_bucket }}/{{ ds }}/orders.csv",
    "target": "gs://{{ var.value.target_bucket }}/{{ dag.dag_id }}/{{ ds }}/summary.csv",
    "bigquery": "{{ var.value.bq_dataset }}.{{ var.value.bq_table }}",
    "partition": "{{ ds_nodash }}",
    "previous": "gs://{{ var.value.source_bucket }}/{{ prev_ds }}/orders.csv",
    "window": "[{{ data_interval_start }}, {{ data_interval_end }})",
}

variables = {
    "source_bucket": "demo-source-bucket",
    "target_bucket": "demo-target-bucket",
    "bq_dataset": "analytics",
    "bq_table": "orders_enriched",
}

ctx = build_context(
    dag_id="etl_to_gcs",
    logical_date="2026-01-15",
    variables=variables,
)

for name, template in TEMPLATES.items():
    print(f"{name:<10} {template}")
    print(f"{'':<10} -> {render_template(template, ctx)}")
    print()

### `template_fields` — the mechanism

Airflow does not render *every* string in your DAG. It renders only the
attributes a task declares as templatable:

```python
class BashOperator(BaseOperator):
    template_fields = ("bash_command", "env")

class BigQueryExecuteQueryOperator(BaseOperator):
    template_fields = ("sql", "impersonation_chain")
```

This is a **class-level declaration**. Each operator type lists its own
templatable attributes. When Airflow runs a task, it walks `template_fields`,
renders each value, and replaces it with the rendered string.

For a custom operator, the whole mechanism is:

```python
class SummaryOperator(BaseOperator):
    template_fields = ("bucket", "prefix")
```

### The rendering-timing trap

Rendering happens **before** the task body runs. That means:

- a template can reference the logical date, Variables, Connections and params;
- a template **cannot** reference a value another task computed — that value
  arrives via XCom, at run time, in the body, not in the template.

```python
# WRONG: 'row_count' is not in the Jinja context.
sql = "SELECT * FROM orders WHERE dt = '{{ ds }}' AND count = {{ row_count }}"

# RIGHT: render the date in the template, pass the XCom value in the body.
sql = "SELECT * FROM orders WHERE dt = '{{ ds }}'"
```

This is why the course's DAGs use `{{ ds }}` for the partition and XCom for
everything computed.

## 4. The undefined-variable failure

This is the single most important behaviour in the whole templating story.

Airflow renders with **strict undefined**. A name that is not in the context
raises — it does **not** silently become an empty string.

Why this matters enormously in production:

```python
bash_command="gsutil cp gs://{{ var.value.source_bucket }}/{{ ds }}/orders.csv ..."
```

If `source_bucket` is misspelled as `source_bukcet`, a permissive renderer
produces:

```bash
gsutil cp gs:///{{ ds }}/orders.csv ...
```

which looks like it might work, and silently writes to the wrong place (or
fails much later, in a confusing way). Strict rendering fails the task
immediately, at the start of the run, with a precise message.

Let's see it.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

from templating import render_template, TemplateRenderError

ctx = build_context(
    dag_id="etl_to_gcs",
    logical_date="2026-01-15",
    variables=variables,
)

# A typo in a Variable name.
bad_template = "gs://{{ var.value.source_bukcet }}/{{ ds }}/orders.csv"

try:
    render_template(bad_template, ctx)
except TemplateRenderError as exc:
    print("TemplateRenderError raised.\n")
    print(f"  template : {exc.template}")
    print(f"  cause    : {type(exc.cause).__name__}")
    print()
    print("The underlying Jinja error:")
    print("  " + str(exc.cause))

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

# The same failure for a completely unknown name.
for template in (
    "{{ not_a_real_variable }}",
    "{{ dag.nonexistent_field }}",
    "{{ params.missing_param }}",
):
    try:
        render_template(template, ctx)
    except TemplateRenderError as exc:
        first_line = str(exc.cause).splitlines()[0]
        print(f"{template:<32} -> TemplateRenderError")
        print(f"{'':<32}    {first_line}")

### `default` and the two ways to be safe

There are exactly two safe patterns, and you should pick one deliberately:

**1. The `default` filter** — use it when a missing value has an obvious
fallback:

```python
"{{ var.value.bq_dataset | default('analytics') }}"
```

The `default` filter only works because the lookup *returns* an undefined
sentinel rather than raising immediately. The course's `_VariableView` is
written to do exactly that — a comment in the source explains why raising
there would bypass `default` entirely.

**2. The `dag_config` layering** — put the fallback in one place:

```python
BQ_DATASET = "{{ var.value.bq_dataset | default('analytics') }}"
```

Both are good. The second is better when many templates need the same
fallback, because there is one source of truth.

Let's verify `default` works, and that it does not mask a *used* undefined
value.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

# 1. default rescues a missing Variable.
print(
    "with default    :",
    render_template("{{ var.value.not_set | default('fallback') }}", ctx),
)

# 2. Without default, the same lookup fails.
try:
    render_template("{{ var.value.not_set }}", ctx)
except TemplateRenderError as exc:
    print(
        "without default : TemplateRenderError -",
        str(exc.cause).splitlines()[0],
    )

# 3. var.get() is the programmatic equivalent.
print("var.get()       :", ctx["__variables__"].get("not_set", "fallback"))

# 4. A missing PARAM is still an error, even with default on the wrong object.
try:
    render_template("{{ var.value.bq_dataset | default('analytics') }}", ctx)
    print("bq_dataset      : rendered fine (it exists)")
except TemplateRenderError as exc:
    print("bq_dataset      :", exc)

## 5. Variables vs params vs Connections

Three different mechanisms, three different lifetimes, three different
audiences. Getting this wrong is one of the most common causes of "works on
my environment" bugs.

| | **Variable** | **param** | **Connection** |
|---|---|---|---|
| **Scope** | environment-wide, across DAGs | one DAG (or one run) | environment-wide, across DAGs |
| **Defined by** | operator / UI | the DAG author | operator / UI / Secret Manager |
| **Overridden by** | operator / UI | whoever triggers the run | operator / UI |
| **Typed** | string (or JSON) | any Python type | structured (host, login, schema, extras) |
| **Holds secrets?** | **No** — visible in the UI | no | **Yes** — the right place for them |
| **Template syntax** | `{{ var.value.x }}` | `{{ params.x }}` | `{{ conn.my_conn.host }}` |
| **Typical use** | bucket names, project IDs, thresholds | backfill date ranges, one-off flags | API credentials |

### The rules that matter

1. **Never put a secret in a Variable.** Variables are visible in the Airflow
   UI to anyone with access to the environment, and they are stored
   unencrypted in the metadata database. Use a **Connection** (ideally backed
   by Secret Manager) for passwords, API keys and service-account JSON.
2. **A Variable is for environment differences.** "The bucket is different in
   prod" → Variable.
3. **A param is for run differences.** "This backfill covers January" → param.
4. **A Connection is for credentials.** Always.

The course's `project/dags/dag_config.py` documents exactly this layering.
Let's look at it and at the defaults the runner ships with.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

import dag_config

print("DEFAULT_VARIABLES:")
for key, value in sorted(local_runner.DEFAULT_VARIABLES.items()):
    print(f"  {key:<22} {value!r}")

print()
print("DEFAULT_CONNECTIONS:")
for name, spec in sorted(local_runner.DEFAULT_CONNECTIONS.items()):
    print(f"  {name}:")
    for field, value in sorted(spec.items()):
        print(f"      {field:<12} {value!r}")

print()
print("The layering, as the course's DAG files express it:\n")
for name in (
    "GCP_PROJECT_ID", "SOURCE_BUCKET", "TARGET_BUCKET",
    "BQ_DATASET", "BQ_TABLE", "FULL_LOAD_THRESHOLD",
):
    print(f"  {name:<20} = {getattr(dag_config, name)!r}")

print()
print("Notice: these are TEMPLATE STRINGS, not values. They are rendered")
print("at run time against the context. That is what makes the same DAG")
print("file work in dev and prod without editing.")

### Rendering the config against two environments

Here is the payoff: the same `dag_config` values render differently for two
environments, with no code change and no file edit.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

from templating import build_context, render_template

DEV_VARS = {
    "gcp_project_id": "demo-project",
    "source_bucket": "demo-source-bucket",
    "target_bucket": "demo-target-bucket",
    "bq_dataset": "analytics",
    "bq_table": "orders_enriched",
}

PROD_VARS = {
    "gcp_project_id": "acme-data-prod",
    "source_bucket": "acme-prod-raw",
    "target_bucket": "acme-prod-curated",
    "bq_dataset": "analytics_prod",
    "bq_table": "orders_enriched",
}

FIELDS = ("GCP_PROJECT_ID", "SOURCE_BUCKET", "TARGET_BUCKET", "BQ_DATASET", "BQ_TABLE")

import dag_config

for label, variables in (("DEV", DEV_VARS), ("PROD", PROD_VARS)):
    ctx = build_context(
        dag_id="etl_to_gcs",
        logical_date="2026-01-15",
        variables=variables,
    )
    print(f"--- {label} ---")
    for field in FIELDS:
        rendered = render_template(getattr(dag_config, field), ctx)
        print(f"  {field:<16} {rendered}")
    print()

### The layering pattern

The course's recommended precedence chain, from lowest to highest priority:

```text
hard-coded default  →  Variable override  →  param override
```

In `dag_config.py`:

```python
BUCKET = "{{ params.bucket | default(var.value.gcs_bucket) }}"
```

Read that as: *use the run's `bucket` param if the operator supplied one;
otherwise fall back to the environment's `gcs_bucket` Variable; otherwise fall
back to the hard-coded default.*

Let's build and verify that chain.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

LAYER = (
    "{{ params.bucket | default(var.value.gcs_bucket | default('hard-coded-bucket')) }}"
)

print("Layer 1 - nothing set:")
ctx = build_context(logical_date="2026-01-15")
print(f"  {render_template(LAYER, ctx)}")
print()

print("Layer 2 - Variable set:")
ctx = build_context(logical_date="2026-01-15", variables={"gcs_bucket": "var-bucket"})
print(f"  {render_template(LAYER, ctx)}")
print()

print("Layer 3 - Variable AND param set (param wins):")
ctx = build_context(
    logical_date="2026-01-15",
    variables={"gcs_bucket": "var-bucket"},
    params={"bucket": "param-bucket"},
)
print(f"  {render_template(LAYER, ctx)}")

### Connections — the credential path

A Connection is exposed as `{{ conn.<conn_id>.<field> }}`. Extras are
available as a parsed dict via `extra_dejson`.

The course's `http_orders_api` connection has extras
`{"timeout": 30, "retries": 3}`. Let's render from it.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from templating import build_context, render_template

ctx = build_context(
    dag_id="etl_to_gcs",
    logical_date="2026-01-15",
    connections=local_runner.DEFAULT_CONNECTIONS,
)

TEMPLATES = [
    "{{ conn.gcp_default.host }}",
    "{{ conn.gcp_default.schema }}",
    "{{ conn.http_orders_api.host }}",
    "{{ conn.http_orders_api.login }}",
    "{{ conn.http_orders_api.extra_dejson.timeout }}",
    "{{ conn.http_orders_api.extra_dejson.retries }}",
    "{{ conn.gcp_default.extra_dejson.location }}",
]

for template in TEMPLATES:
    print(f"  {template:<52} -> {render_template(template, ctx)}")
print()
print("Note: conn.http_orders_api.password renders too, which is exactly")
print("why you must not put it in a log line or a templated bash command.")

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from templating import TemplateRenderError

ctx = build_context(
    dag_id="etl_to_gcs",
    logical_date="2026-01-15",
    connections=local_runner.DEFAULT_CONNECTIONS,
)

# A missing connection raises, like everything else.
try:
    render_template("{{ conn.no_such_conn.host }}", ctx)
except TemplateRenderError as exc:
    print("Missing connection -> TemplateRenderError")
    print("  " + str(exc.cause).splitlines()[0])

# A missing FIELD on an existing connection also raises.
try:
    render_template("{{ conn.gcp_default.password }}", ctx)
except TemplateRenderError as exc:
    print()
    print("Missing field on an existing connection -> TemplateRenderError")
    print("  " + str(exc.cause).splitlines()[0])

## 6. Rendering in a real task

Let's put the whole thing together: a DAG whose tasks render real GCS
paths, and confirm the rendered path matches what `pipeline_logic.gcs_object_path`
builds.

This is a genuine consistency check you can put in a test: *the Jinja path and
the programmatic path must agree.*

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import gcs_object_path

dag = local_runner.DAGS["templating_paths"]()
run = local_runner.run_dag(
    dag,
    logical_date="2026-01-15",
    variables=local_runner.DEFAULT_VARIABLES,
    connections=local_runner.DEFAULT_CONNECTIONS,
)

rendered = run.pull("render_paths")
print("Rendered by the templating DAG:")
for key, value in rendered.items():
    print(f"  {key:<10} {value}")
print()
print("States:", run.state)

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

# Cross-check: the templated path equals the programmatically-built path.
templated = rendered["target"]
programmatic = gcs_object_path(
    local_runner.DEFAULT_VARIABLES["target_bucket"],
    "etl_to_gcs",
    f"{run.logical_date}/summary.csv",
)

print(f"  templated     : {templated}")
print(f"  programmatic  : {programmatic}")
print(f"  agree?        : {templated == programmatic}")
print()
print("The two builders compose the path differently: the templated form")
print("puts the date as its own path segment, while gcs_object_path treats")
print("the prefix as one path. That is exactly why you test such pairs")
print("rather than assume they agree.")
print()
print(f"  templated target    : {rendered['target']}")
print(f"  templated partition : {rendered['partition']}")
print(f"  templated source    : {rendered['source']}")

## 7. Debugging a template, the practical workflow

When a template misbehaves in Composer, the debugging order is:

1. **Render it locally.** In the Composer UI, the task log shows the
   *rendered* command under "Rendered Template". Reproduce it locally with
   `render_template`.
2. **Print the context.** `build_context(...)` and inspect the keys. The
   context is a plain dict — print it.
3. **Check the Variable exists.** `{{ var.value.x }}` where `x` is not set
   raises. Verify with `var.value.get("x")`.
4. **Check strictness.** Did you use `default` where you meant to?
5. **Check `template_fields`.** If your custom operator's attribute is not
   listed, it is never rendered at all — no error, just the raw `{{ ... }}`
   string in the log. This is the most confusing failure mode, because
   nothing is wrong with the template itself.

That last one is worth stating twice: **an unrendered template is not an
error, it's a literal string.** If you see `{{ ds }}` in a log line, look at
`template_fields`, not at the context.

## 8. The complete context, rendered

One cell that shows every context variable at once, for a specific run. This
is the reference you can come back to.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from templating import build_context

ctx = build_context(
    dag_id="etl_to_gcs",
    task_id="aggregate",
    logical_date="2026-01-15",
    run_id="scheduled__2026-01-15T00:00:00+00:00",
    variables=local_runner.DEFAULT_VARIABLES,
    connections=local_runner.DEFAULT_CONNECTIONS,
    params={"audience": "data-engineers"},
)

print("Every key in the context:\n")
for key in sorted(ctx):
    value = ctx[key]
    if key.startswith("__"):
        print(f"  {key:<22} <internal: {len(value)} entries>")
    elif isinstance(value, dict):
        print(f"  {key:<22} {value}")
    else:
        print(f"  {key:<22} {value!r}")

## Exercises

1. **A template that must not fail.** Write a template that renders the target bucket, the logical date and the DAG id, and that falls back to a default bucket when the Variable is missing. Render it against a context with and without the Variable.
2. **Harder — detect the unrendered template.** Airflow only renders attributes listed in `template_fields`. Write a small function `is_rendered(value)` that returns `False` when a string still contains `{{` or `}}`. Apply it to the rendered output of the `templating_paths` DAG for every path, and explain what it would mean if one of them came back `False`.
3. **Harder — params vs Variables.** Build two contexts for the same logical date: one where `target_bucket` is a Variable and one where it is a param. Use the layering template `{{ params.target_bucket | default(var.value.target_bucket) }}`. Show that the param wins, and explain which one a *backfill trigger* should use and why.

<details>
<summary><b>Solution (click to reveal)</b></summary>

```python
# Exercise 1
TEMPLATE = (
    "gs://{{ var.value.target_bucket | default('fallback-bucket') }}"
    "/{{ dag.dag_id }}/{{ ds }}/summary.csv"
)

ctx_no_var = build_context(dag_id="etl_to_gcs", logical_date="2026-01-15")
print(render_template(TEMPLATE, ctx_no_var))
# gs://fallback-bucket/etl_to_gcs/2026-01-15/summary.csv

ctx_with_var = build_context(
    dag_id="etl_to_gcs",
    logical_date="2026-01-15",
    variables={"target_bucket": "real-bucket"},
)
print(render_template(TEMPLATE, ctx_with_var))
# gs://real-bucket/etl_to_gcs/2026-01-15/summary.csv
```

```python
# Exercise 2
def is_rendered(value):
    return isinstance(value, str) and "{{" not in value and "}}" not in value

for key, value in rendered.items():
    print(f"{key:<10} rendered={is_rendered(value)}  {value}")
```

If a value comes back `False`, the template string was never rendered at
all — it reached the task body as a literal string. The cause is almost
always that the attribute holding it is not listed in the operator's
`template_fields`, so Airflow skipped it. It is *not* a context problem: a
context problem would raise a `TemplateRenderError`, not silently pass the
raw `{{ ... }}` through.

```python
# Exercise 3
LAYER = "{{ params.target_bucket | default(var.value.target_bucket) }}"

ctx_var_only = build_context(
    logical_date="2026-01-15", variables={"target_bucket": "var-bucket"}
)
ctx_both = build_context(
    logical_date="2026-01-15",
    variables={"target_bucket": "var-bucket"},
    params={"target_bucket": "param-bucket"},
)

print(render_template(LAYER, ctx_var_only))   # var-bucket
print(render_template(LAYER, ctx_both))       # param-bucket
```

A backfill trigger should use a **param**, because the target for a backfill
is a property of *that run* ("write this to the staging bucket"), not of the
environment. A Variable is for "which bucket does this environment use",
which is stable across runs. Using a Variable for a backfill destination
means every backfill run and the live schedule write to the same place,
which is how you end up with a backfill that truncates the live table.

</details>